# 🎨 Poster Quality Scorer — TOPSIS Method
> **Improved & fixed version.** Trains on a poster dataset, then scores any new image consistently using saved reference points.

In [ ]:
# CELL 1 — Install dependencies (run once in Colab)
!pip install scikit-image torchvision opencv-contrib-python pytesseract pillow
!apt-get install -y tesseract-ocr -q

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [ ]:
# CELL 2 — Imports
import os, pickle
import cv2
import numpy as np
import pandas as pd
import torch
from torchvision import models, transforms
from PIL import Image
from skimage.measure import shannon_entropy
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge

In [ ]:
# CELL 3 — Load ResNet50 (feature extractor, run once)
resnet = models.resnet50(pretrained=True)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])
resnet.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
print("Model loaded.")

Model loaded.


In [ ]:
# CELL 4 — Feature extraction helpers

NON_CNN_FEATS = [
    "brightness", "contrast", "entropy", "edge_density",
    "hue_diversity",          # ← replaces raw colorfulness (fixes false "too many colors")
    "lr_balance", "tb_balance",
    "whitespace", "saliency", "text_density", "word_count"
]

def hue_diversity_score(img_rgb):
    """
    Count how many DISTINCT hues are present (robust to saturation noise).
    Returns 0-100 where:
      < 20 → monochromatic / very limited palette
      20-50 → limited palette (e.g. red + white + one accent)
      > 60 → genuinely many different hues
    Raw colorfulness (|R-G| + |YB|) spikes on red+yellow-green pairs even
    with only 3 hues, causing false "too many colors" flags. Hue diversity
    counts actual hue clusters, not chroma magnitude.
    """
    hsv  = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    mask = hsv[:, :, 1] > 40          # ignore near-grey pixels (low saturation)
    if mask.sum() < 100:
        return 0.0                     # essentially greyscale poster
    hues = hsv[:, :, 0][mask]         # H channel: 0-179 in OpenCV
    hist, _ = np.histogram(hues, bins=36, range=(0, 180))  # 5° bins
    occupied = np.sum(hist > (mask.sum() * 0.005))  # bins with >0.5% of pixels
    return float(min(occupied * 3, 100))             # scale to 0-100

def extract_features(img_path):
    """Extract all features from a poster image. Returns dict."""
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        raise ValueError(f"Cannot read: {img_path}")

    img  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape

    # --- Low-level ---
    brightness  = float(np.mean(gray))
    contrast    = float(np.std(gray))
    entropy_val = float(shannon_entropy(gray))

    edges       = cv2.Canny(gray, 100, 200)
    edge_dens   = float(np.sum(edges > 0) / edges.size)

    hue_div     = hue_diversity_score(img)   # ← replaces raw colorfulness

    # --- Layout balance (normalised 0-1) ---
    gf          = gray.astype(np.float64)
    left, right = gf[:, :w//2].sum(), gf[:, w//2:].sum()
    top, bottom = gf[:h//2, :].sum(), gf[h//2:, :].sum()
    lr_balance  = abs(left  - right)  / (left  + right  + 1e-6)
    tb_balance  = abs(top   - bottom) / (top   + bottom + 1e-6)

    # --- Whitespace ---
    _, thresh   = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY)
    whitespace  = float(np.sum(thresh == 255) / thresh.size)

    # --- Saliency (Laplacian proxy) ---
    saliency    = float(np.mean(np.abs(cv2.Laplacian(gray, cv2.CV_64F))))

    # --- Text (OCR) ---
    try:
        import pytesseract
        text         = pytesseract.image_to_string(Image.fromarray(gray))
        words        = text.split()
        word_count   = len(words)
        text_density = word_count / (h * w / 1000.0 + 1e-6)
    except Exception:
        word_count, text_density = 0, 0.0

    # --- CNN (ResNet50 pool, 2048-d) ---
    tensor = transform(Image.fromarray(img)).unsqueeze(0)
    with torch.no_grad():
        cnn_feat = resnet(tensor).squeeze().numpy()   # (2048,)

    row = {
        "brightness": brightness, "contrast": contrast, "entropy": entropy_val,
        "edge_density": edge_dens, "hue_diversity": hue_div,
        "lr_balance": lr_balance, "tb_balance": tb_balance,
        "whitespace": whitespace, "saliency": saliency,
        "text_density": text_density, "word_count": word_count,
    }
    for i, v in enumerate(cnn_feat):
        row[f"cnn_{i}"] = float(v)
    return row

In [ ]:
# CELL 5 — Build dataset from poster folder
import zipfile

zip_path   = "/content/posters.zip"     # ← path to your zip
extract_to = "posters_folder"

os.makedirs(extract_to, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_to)

# Auto-detect poster folder (handles nested structure)
DATASET_PATH = extract_to
for item in os.listdir(extract_to):
    candidate = os.path.join(extract_to, item)
    if os.path.isdir(candidate):
        DATASET_PATH = candidate
        break

print("Poster folder:", DATASET_PATH)
print("Files:", os.listdir(DATASET_PATH)[:5])

Poster folder: posters_folder/posters
Files: ['Screenshot 2026-04-07 223649.png', 'Screenshot 2026-04-07 234959.png', 'Screenshot 2026-04-08 002421.png', 'Screenshot 2026-04-07 235552.png', 'Screenshot 2026-04-08 004848.png']


In [ ]:
# CELL 6 — Extract features for all posters
rows = []
for fname in os.listdir(DATASET_PATH):
    fpath = os.path.join(DATASET_PATH, fname)
    try:
        feat = extract_features(fpath)
        feat["file"] = fname
        rows.append(feat)
        print(f"  ✓ {fname}")
    except Exception as e:
        print(f"  ✗ {fname}: {e}")

df = pd.DataFrame(rows)
print(f"\nDataset shape: {df.shape}")
df[["file"] + NON_CNN_FEATS].head()

  ✓ Screenshot 2026-04-07 223649.png
  ✓ Screenshot 2026-04-07 234959.png
  ✓ Screenshot 2026-04-08 002421.png
  ✓ Screenshot 2026-04-07 235552.png
  ✓ Screenshot 2026-04-08 004848.png
  ✓ 7t.jpg.jpeg
  ✓ Screenshot 2026-04-08 003846.png
  ✓ Screenshot 2026-04-08 002829.png
  ✓ Screenshot 2026-04-07 222934.png
  ✓ Screenshot 2026-04-08 000228.png
  ✓ Screenshot 2026-04-08 002705.png
  ✓ Screenshot 2026-04-07 234939.png
  ✓ Screenshot 2026-04-08 004704.png
  ✓ Screenshot 2026-04-07 235044.png
  ✓ Screenshot 2026-04-08 002332.png
  ✓ Screenshot 2026-04-08 002149.png
  ✓ Screenshot 2026-04-08 004634.png
  ✓ Screenshot 2026-04-08 004832.png
  ✓ Screenshot 2026-04-08 004905.png
  ✓ Screenshot 2026-04-08 002250.png
  ✓ Screenshot 2026-04-08 005318.png
  ✓ Screenshot 2026-04-08 000207.png
  ✓ Screenshot 2026-04-07 234917.png
  ✓ Screenshot 2026-04-08 002342.png
  ✓ Screenshot 2026-04-08 004319.png
  ✓ Screenshot 2026-04-07 224715.png
  ✓ 8t.jpg.jpeg
  ✓ Screenshot 2026-04-08 005238.png
  ✓ Sc

,file,brightness,contrast,entropy,edge_density,hue_diversity,lr_balance,tb_balance,whitespace,saliency,text_density,word_count
0,Screenshot 2026-04-07 223649.png,134.514063,58.184751,7.635954,0.062205,27.0,0.067042,0.162287,0.013635,9.838552,0.035794,36
1,Screenshot 2026-04-07 234959.png,214.753804,57.692326,3.857000,0.054627,42.0,0.035124,0.098251,0.706014,8.183626,0.033302,31
2,Screenshot 2026-04-08 002421.png,232.799772,44.493428,5.300625,0.071308,18.0,0.000784,0.001433,0.727620,18.271086,0.258430,169
3,Screenshot 2026-04-07 235552.png,214.231882,63.331611,5.031180,0.024193,21.0,0.018855,0.111828,0.662836,5.995249,0.022591,29
4,Screenshot 2026-04-08 004848.png,191.782253,78.049912,6.674073,0.072933,9.0,0.123182,0.034892,0.483208,15.940452,0.057637,26


In [ ]:
# CELL 7 — PCA on CNN features only
PCA_N    = 20
cnn_cols = [c for c in df.columns if c.startswith("cnn_")]

X_non_cnn = df[NON_CNN_FEATS].copy().astype(float)
X_cnn     = df[cnn_cols].values.astype(float)

pca_cnn   = PCA(n_components=PCA_N)
cnn_red   = pca_cnn.fit_transform(X_cnn)

X_new = X_non_cnn.copy()
for i in range(PCA_N):
    X_new[f"cnn_pca_{i}"] = cnn_red[:, i]

print(f"Explained variance by {PCA_N} CNN PCs: {pca_cnn.explained_variance_ratio_.sum():.3f}")
print("Feature matrix:", X_new.shape)

Explained variance by 20 CNN PCs: 0.704
Feature matrix: (80, 31)


In [ ]:
# CELL 8 — Ideal-based transform (make all features "higher is better")
# hue_diversity replaces colorfulness here.
# Ideal for hue_diversity is ~25 (limited palette = intentional, good design).
OPTIMAL_FEATS   = ["contrast", "entropy", "hue_diversity", "whitespace", "text_density"]
optimal_medians = {c: float(X_new[c].median()) for c in OPTIMAL_FEATS}

X_adj = X_new.copy()
for col, med in optimal_medians.items():
    X_adj[col] = -np.abs(X_new[col] - med)

print("Optimal medians (training):")
for k, v in optimal_medians.items():
    print(f"  {k:<20} {v:.4f}")

Optimal medians (training):
  contrast             66.4605
  entropy              5.8088
  hue_diversity        33.0000
  whitespace           0.3442
  text_density         0.0526


In [ ]:
# CELL 9 — StandardScaler
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_adj)
print("Scaled shape:", X_scaled.shape)

Scaled shape: (80, 31)


In [ ]:
# CELL 10 — Data-driven weights via Ridge regression proxy
# Step 1: equal-weight TOPSIS → proxy target
w0   = np.ones(X_scaled.shape[1]) / X_scaled.shape[1]
Xw0  = X_scaled * w0
ib0  = Xw0.max(axis=0); iw0 = Xw0.min(axis=0)
db0  = np.linalg.norm(Xw0 - ib0, axis=1)
dw0  = np.linalg.norm(Xw0 - iw0, axis=1)
y_proxy = dw0 / (db0 + dw0 + 1e-9)

# Step 2: Ridge importance → weights
ridge = Ridge(alpha=1.0)
ridge.fit(X_scaled, y_proxy)
weights = np.abs(ridge.coef_)
weights = weights / (weights.sum() + 1e-9)

feat_names = list(X_adj.columns)
top_idx = np.argsort(weights)[::-1][:10]
print("Top 10 features by weight:")
for i in top_idx:
    print(f"  {feat_names[i]:<25} {weights[i]:.4f}")

Top 10 features by weight:
  cnn_pca_18                0.0415
  cnn_pca_6                 0.0385
  cnn_pca_13                0.0382
  cnn_pca_12                0.0371
  word_count                0.0355
  cnn_pca_1                 0.0351
  cnn_pca_8                 0.0348
  cnn_pca_15                0.0346
  cnn_pca_5                 0.0346
  cnn_pca_17                0.0342


In [ ]:
# CELL 11 — TOPSIS scoring (training set)
X_weighted     = X_scaled * weights

# ⚡ Save reference points — REQUIRED for consistent new-image scoring
ideal_best_ref  = X_weighted.max(axis=0)
ideal_worst_ref = X_weighted.min(axis=0)

d_best  = np.linalg.norm(X_weighted - ideal_best_ref, axis=1)
d_worst = np.linalg.norm(X_weighted - ideal_worst_ref, axis=1)
raw     = d_worst / (d_best + d_worst + 1e-9)

# Save normalisation range too
score_min, score_max = raw.min(), raw.max()
df["topsis_score"] = 100 * (raw - score_min) / (score_max - score_min + 1e-9)

print(df[["file","topsis_score"]].sort_values("topsis_score", ascending=False).to_string(index=False))

                                         file  topsis_score
             Screenshot 2026-04-07 234917.png     99.999999
             Screenshot 2026-04-08 003339.png     72.834289
             Screenshot 2026-04-08 004453.png     71.878161
             Screenshot 2026-04-08 002250.png     69.411823
             Screenshot 2026-04-08 002611.png     66.624686
             Screenshot 2026-04-08 004848.png     63.222291
             Screenshot 2026-04-08 002829.png     61.195354
             Screenshot 2026-04-07 224645.png     60.109535
                                  7t.jpg.jpeg     56.292971
             Screenshot 2026-04-08 002342.png     55.819417
             Screenshot 2026-04-08 003546.png     51.699103
             Screenshot 2026-04-08 005238.png     51.054816
             Screenshot 2026-04-08 002226.png     50.476351
             Screenshot 2026-04-08 004319.png     50.455216
             Screenshot 2026-04-08 004634.png     50.074520
             Screenshot 2026-04-08 00513

In [ ]:
# CELL 12 — Issue detection aligned with scoring weights
# FIX: thresholds now scale with each feature's Ridge weight so that
# features that drive TOPSIS down also surface as issues.
# Low-weight features use a loose threshold (z > ±1.5),
# high-weight features use a tighter threshold (z > ±0.6).

stats = {c: {"mean": float(X_new[c].mean()), "std": float(X_new[c].std())}
         for c in NON_CNN_FEATS}

def _threshold_for(col):
    """Tighter threshold for high-weight features, looser for low-weight ones."""
    idx = feat_names.index(col) if col in feat_names else None
    if idx is None:
        return 1.0
    w = weights[idx]
    w_max = weights[:len(NON_CNN_FEATS)].max()
    # linearly map weight → threshold between 0.6 (high weight) and 1.5 (low weight)
    t = 1.5 - 0.9 * (w / (w_max + 1e-9))
    return float(np.clip(t, 0.6, 1.5))

def detect_issues(row_dict):
    """Return list of design issues for a feature dict."""
    issues = []
    def z(col):
        return (row_dict[col] - stats[col]["mean"]) / (stats[col]["std"] + 1e-6)

    t_contrast  = _threshold_for("contrast")
    t_entropy   = _threshold_for("entropy")
    t_edge      = _threshold_for("edge_density")
    t_white     = _threshold_for("whitespace")
    t_hue       = _threshold_for("hue_diversity")
    t_text      = _threshold_for("text_density")
    t_bal       = _threshold_for("lr_balance")

    if z("contrast")      < -t_contrast:  issues.append("Low contrast – text may be hard to read")
    if z("contrast")      >  2.0:         issues.append("Overly harsh contrast")
    if z("entropy")       >  t_entropy:   issues.append("Too cluttered – reduce visual elements")
    if z("entropy")       < -t_entropy:   issues.append("Too simple – lacks visual interest")
    if z("edge_density")  >  t_edge:      issues.append("Too many edges – cluttered layout")
    if z("whitespace")    < -t_white:     issues.append("Not enough whitespace – cramped design")

    # hue_diversity: only flag "too many colors" when genuinely diverse palette
    # (hue_diversity > 60 maps to z >> 0 in most datasets)
    if z("hue_diversity") >  t_hue and row_dict["hue_diversity"] > 60:
        issues.append("Too many colors – may look inconsistent")
    if z("hue_diversity") < -t_hue and row_dict["hue_diversity"] < 10:
        issues.append("Dull colors – lacks visual appeal")

    if z("text_density")  >  t_text:      issues.append("Too much text on poster")
    if z("text_density")  < -t_text:      issues.append("Too little text – add more info")
    if z("lr_balance")    >  t_bal or z("tb_balance") > t_bal:
        issues.append("Layout imbalance – elements unevenly distributed")

    return issues

records = X_new[NON_CNN_FEATS].to_dict("records")
df["issues"] = [detect_issues(r) for r in records]

# Sanity check: low scorers should now show issues
low = df.nsmallest(5, "topsis_score")[["file","topsis_score","issues"]]
print("Bottom 5 posters:")
print(low.to_string(index=False))

Bottom 5 posters:
                                         file  topsis_score                                                                                                         issues
                                  4t.jpg.jpeg      0.000000 [Low contrast – text may be hard to read, Too simple – lacks visual interest, Too little text – add more info]
             Screenshot 2026-04-07 224029.png      1.218031                               [Too cluttered – reduce visual elements, Not enough whitespace – cramped design]
WhatsApp Image 2026-04-07 at 11.03.47 PM.jpeg      5.196586                                  [Low contrast – text may be hard to read, Too simple – lacks visual interest]
             Screenshot 2026-04-08 002457.png      8.786427                                                                      [Low contrast – text may be hard to read]
             Screenshot 2026-04-08 003846.png      9.827142                               [Too cluttered – reduce visual elemen

In [ ]:
# CELL 13 — Save all trained state (run AFTER training)
state = {
    "pca_cnn":          pca_cnn,
    "scaler":           scaler,
    "weights":          weights,
    "ideal_best_ref":   ideal_best_ref,
    "ideal_worst_ref":  ideal_worst_ref,
    "score_min":        score_min,
    "score_max":        score_max,
    "optimal_medians":  optimal_medians,
    "stats":            stats,
    "feat_names":       feat_names,        # needed for weight-aligned thresholds
}

with open("poster_scorer_state.pkl", "wb") as f:
    pickle.dump(state, f)
print("Saved: poster_scorer_state.pkl")

Saved: poster_scorer_state.pkl


In [ ]:
# CELL 14 — Score a NEW poster (both fixes applied)
def evaluate_new_poster(image_path, state_path="poster_scorer_state.pkl"):
    """
    Score a single new poster against the trained TOPSIS model.
    Fixes applied:
      1. hue_diversity replaces raw colorfulness → no false "too many colors"
      2. issue thresholds scale with Ridge weights → low scorers show real issues
    """
    with open(state_path, "rb") as f:
        st = pickle.load(f)

    feat = extract_features(image_path)

    # Non-CNN feature vector
    X_non = pd.DataFrame([{c: feat[c] for c in NON_CNN_FEATS}])

    # CNN → PCA (transform only, no re-fit)
    cnn_arr = np.array([feat[f"cnn_{i}"] for i in range(2048)]).reshape(1, -1)
    cnn_red = st["pca_cnn"].transform(cnn_arr)

    X_single = X_non.copy()
    for i in range(cnn_red.shape[1]):
        X_single[f"cnn_pca_{i}"] = cnn_red[0, i]

    # Ideal-based transform using TRAINING medians
    X_adj_s = X_single.copy()
    for col, med in st["optimal_medians"].items():
        X_adj_s[col] = -abs(float(X_single[col].iloc[0]) - med)

    # Scale using TRAINING scaler
    X_sc = st["scaler"].transform(X_adj_s)
    X_wt = X_sc * st["weights"]

    # TOPSIS distances against TRAINING reference points
    d_best_  = float(np.linalg.norm(X_wt - st["ideal_best_ref"]))
    d_worst_ = float(np.linalg.norm(X_wt - st["ideal_worst_ref"]))
    raw      = d_worst_ / (d_best_ + d_worst_ + 1e-9)

    score = 100.0 * (raw - st["score_min"]) / (st["score_max"] - st["score_min"] + 1e-9)
    score = float(np.clip(score, 0, 100))

    # Issue detection with weight-aligned thresholds
    _feat_names = st["feat_names"]
    _weights    = st["weights"]

    def _threshold_for(col):
        idx = _feat_names.index(col) if col in _feat_names else None
        if idx is None: return 1.0
        w     = _weights[idx]
        w_max = _weights[:len(NON_CNN_FEATS)].max()
        return float(np.clip(1.5 - 0.9 * (w / (w_max + 1e-9)), 0.6, 1.5))

    def z(col):
        return (feat[col] - st["stats"][col]["mean"]) / (st["stats"][col]["std"] + 1e-6)

    issues = []
    if z("contrast")     < -_threshold_for("contrast"):    issues.append("Low contrast – text may be hard to read")
    if z("contrast")     >  2.0:                           issues.append("Overly harsh contrast")
    if z("entropy")      >  _threshold_for("entropy"):     issues.append("Too cluttered – reduce visual elements")
    if z("entropy")      < -_threshold_for("entropy"):     issues.append("Too simple – lacks visual interest")
    if z("edge_density") >  _threshold_for("edge_density"):issues.append("Too many edges – cluttered layout")
    if z("whitespace")   < -_threshold_for("whitespace"):  issues.append("Not enough whitespace – cramped design")
    if z("hue_diversity") > _threshold_for("hue_diversity") and feat["hue_diversity"] > 60:
        issues.append("Too many colors – may look inconsistent")
    if z("hue_diversity") < -_threshold_for("hue_diversity") and feat["hue_diversity"] < 10:
        issues.append("Dull colors – lacks visual appeal")
    if z("text_density") >  _threshold_for("text_density"): issues.append("Too much text on poster")
    if z("text_density") < -_threshold_for("text_density"): issues.append("Too little text – add more info")
    if z("lr_balance")   > _threshold_for("lr_balance") or z("tb_balance") > _threshold_for("tb_balance"):
        issues.append("Layout imbalance – elements unevenly distributed")

    return {
        "score":    round(score, 2),
        "issues":   issues,
        "features": {k: round(feat[k], 4) for k in NON_CNN_FEATS},
    }

In [ ]:
# CELL 15 — Example: score a new poster
result = evaluate_new_poster("/content/Screenshot 2026-04-22 105529.png")   # ← change path

print(f"\n{'='*45}")
print(f"  POSTER SCORE:  {result['score']} / 100")
print(f"{'='*45}")

print("\nDesign Issues:")
if result["issues"]:
    for issue in result["issues"]:
        print(f"  ⚠  {issue}")
else:
    print("  ✓  No major issues detected")

print("\nKey Feature Values:")
for k, v in result["features"].items():
    print(f"  {k:<20} {v}")

ValueError: Cannot read: /content/Screenshot 2026-04-22 105529.png